**Nombre:** Daniel Soto Villada

**Cédula:** 1000534026

**Correo Institucional:** daniel.soto1@udea.edu.co

**Correo personal:** dnsv.001@gmail.com

# Problema 14: Plano invariable del sistema Sol–Júpiter–Saturno

Encuentre los vectores de estado del Sol, Júpiter y Saturno medidos respecto al baricentro del sistema solar (origen inercial) para el día lunes **6 de abril de 2026 a las 22:00:00 UT** y, con base en esto:

**a)** Encuentre la inclinación del plano invariable de Laplace del sistema Sol–Júpiter–Saturno con respecto al plano de la eclíptica.

**b)** Compare la ubicación de este plano invariable con la ubicación de los planos de las órbitas de Júpiter y Saturno, sabiendo que la inclinación de sus órbitas (medidas respecto a la eclíptica) son respectivamente $1^{\circ}18'$ y $2^{\circ}29'$.

## Marco teórico

El **plano invariable de Laplace** de un sistema de $N$ cuerpos es el plano perpendicular al momentum angular total del sistema:

$$\vec{L}_{\text{total}} = \sum_{i=1}^{N} m_i\,\vec{R}_i \times \vec{V}_i$$

La **inclinación** de este plano respecto al plano de la eclíptica se obtiene calculando el ángulo entre el vector $\vec{L}_{\text{total}}$ y la normal a la eclíptica $\hat{n}_{\text{ecl}} = (0, 0, 1)$ (en coordenadas eclípticas J2000):

$$i_{\text{inv}} = \arccos\!\left(\frac{L_z}{|\vec{L}_{\text{total}}|}\right)$$

De manera análoga, la inclinación del plano orbital de un cuerpo respecto a la eclíptica es:

$$i_k = \arccos\!\left(\frac{L_{k,z}}{|\vec{L}_k|}\right)$$

y el ángulo entre el plano invariable y el plano orbital del cuerpo $k$ es:

$$\Delta\theta_k = \arccos\!\left(\hat{L}_{\text{total}} \cdot \hat{L}_k\right)$$

In [ ]:
#@title Instalación de paquetes
!pip install -Uq pymcel

In [ ]:
#@title Importación de librerías
import numpy as np
import pymcel as pc
import spiceypy as spy

In [ ]:
#@title Descarga de kernels
pc.descarga_kernels()
spy.furnsh([
    'pymcel/data/gm_de431.tpc',
    'pymcel/data/de430.bsp',
    'pymcel/data/latest_leapseconds.tls'
])

In [ ]:
#@title Obtención de vectores de estado con `pymcel`

# Cuerpos del sistema Sol–Júpiter–Saturno
bodies = [
    'Sun',
    'Jupiter Barycenter',
    'Saturn Barycenter'
]

# Fecha del problema: lunes 6 de abril de 2026 a las 22:00:00 UT
FECHA = '2026-04-06 22:00:00'

datoss = []   # [x, y, z, vx, vy, vz] en unidades del SI: m y m/s
muCuerpos = []

for id in bodies:
    tabla, df, datos = pc.consulta_horizons(
        id=id,
        location='@0',    # Baricentro del Sistema Solar (SSB)
        epochs=FECHA,
        datos='vectors'
    )
    datoss.append(datos)
    muCuerpos.append(spy.bodvrd(id, 'GM', 1)[1][0])

In [ ]:
#@title Vectores de posición y velocidad

N = len(bodies)  # 3 cuerpos: Sol, Júpiter, Saturno

Rs = np.zeros((N, 3))  # Posiciones [m]
Vs = np.zeros((N, 3))  # Velocidades [m/s]

for i in range(N):
    Rs[i] = datoss[i][0:3]
    Vs[i] = datoss[i][3:]

print("Vectores de estado (posición en m, velocidad en m/s) respecto al SSB:")
print(f"{'Cuerpo':<25} {'x [m]':>18} {'y [m]':>18} {'z [m]':>18}")
for i, id in enumerate(bodies):
    print(f"{id:<25} {Rs[i,0]:>18.3e} {Rs[i,1]:>18.3e} {Rs[i,2]:>18.3e}")
print()
print(f"{'Cuerpo':<25} {'vx [m/s]':>18} {'vy [m/s]':>18} {'vz [m/s]':>18}")
for i, id in enumerate(bodies):
    print(f"{id:<25} {Vs[i,0]:>18.6f} {Vs[i,1]:>18.6f} {Vs[i,2]:>18.6f}")

Obtención de la masa en kg a partir de $GM$:

$$M = \frac{GM \times 10^{9}}{G} \quad [\text{kg}]$$

donde $GM$ viene en $\text{km}^3/\text{s}^2$ desde SPICE, por lo que hay que multiplicar por $10^9$ para convertir a $\text{m}^3/\text{s}^2$.

In [ ]:
#@title Masas a partir de $GM$
G = 6.67191e-11  # Constante de Gravitación Universal [N·m²/kg²] (valor usado en retoComputacional1.ipynb)
Ms = np.array(muCuerpos) * 1e9 / G  # Masas [kg]

print("Masas del sistema Sol–Júpiter–Saturno:")
for i, id in enumerate(bodies):
    print(f"  M[{id}] = {Ms[i]:.6e} kg")

## Cálculo del momentum angular total del sistema Sol–Júpiter–Saturno

$$\vec{L}_i = m_i\,(\vec{R}_i \times \vec{V}_i), \qquad \vec{L}_{\text{total}} = \sum_{i} \vec{L}_i$$

In [ ]:
#@title Momentum angular total

L = np.array([Ms[i] * np.cross(Rs[i], Vs[i]) for i in range(N)])
Lt = L.sum(axis=0)
LtNorm = np.linalg.norm(Lt)
Lt_hat = Lt / LtNorm

print("Momentum angular de cada cuerpo [kg·m²/s]:")
for i, id in enumerate(bodies):
    print(f"  L[{id}] = ({L[i,0]:.3e}, {L[i,1]:.3e}, {L[i,2]:.3e})")

print()
print("Momentum angular total del sistema Sol–Júpiter–Saturno [kg·m²/s]:")
print(f"  Lx = {Lt[0]:.6e}")
print(f"  Ly = {Lt[1]:.6e}")
print(f"  Lz = {Lt[2]:.6e}")
print(f"  |L| = {LtNorm:.6e}")

## a) Inclinación del plano invariable respecto a la eclíptica

En el sistema de coordenadas de la eclíptica J2000, la normal al plano de la eclíptica es $\hat{n}_{\text{ecl}} = (0, 0, 1)$. Por tanto, la inclinación del plano invariable es:

$$\boxed{i_{\text{inv}} = \arccos\!\left(\frac{L_z}{|\vec{L}_{\text{total}}|}\right)}$$

In [ ]:
#@title Inclinación del plano invariable respecto a la eclíptica

# Normal a la eclíptica en coordenadas eclípticas J2000
n_ecl = np.array([0.0, 0.0, 1.0])

# Inclinación del plano invariable respecto a la eclíptica
i_inv_rad = np.arccos(np.dot(Lt_hat, n_ecl))
i_inv_deg = np.rad2deg(i_inv_rad)

# Conversión a grados, minutos, segundos
grados = int(i_inv_deg)
minutos = int((i_inv_deg - grados) * 60)
segundos = ((i_inv_deg - grados) * 60 - minutos) * 60

print("a) Inclinación del plano invariable de Laplace (Sol–Júpiter–Saturno)")
print(f"   respecto al plano de la eclíptica:")
print()
print(f"   i_inv = {i_inv_deg:.6f}°")
print(f"   i_inv = {grados}°{minutos}'{segundos:.2f}''")

## b) Comparación con los planos orbitales de Júpiter y Saturno

Sabemos que las inclinaciones orbitales respecto a la eclíptica son:
- Júpiter: $i_J = 1^\circ 18' = 1.3^\circ$
- Saturno: $i_S = 2^\circ 29' \approx 2.483^\circ$

Además, calculamos la inclinación de cada plano orbital a partir de los vectores de estado (usando $L_{k,z}/|\vec{L}_k|$) para verificar consistencia, y el ángulo $\Delta\theta_k$ entre el plano invariable y el plano orbital de cada planeta.

In [ ]:
#@title Comparación con los planos orbitales de Júpiter y Saturno

# Inclinaciones orbitales dadas (respecto a la eclíptica)
i_Jup_dado_deg = 1 + 18/60          # 1°18'
i_Sat_dado_deg = 2 + 29/60          # 2°29'

# Inclinaciones calculadas a partir de los vectores de estado
# (solo para Júpiter y Saturno, índices 1 y 2)
nombres = ['Júpiter', 'Saturno']
i_calc = []
delta_theta = []

for k in [1, 2]:  # índices de Júpiter y Saturno
    L_k = L[k]
    L_k_norm = np.linalg.norm(L_k)
    L_k_hat = L_k / L_k_norm

    # Inclinación del plano orbital respecto a la eclíptica
    i_k = np.rad2deg(np.arccos(np.dot(L_k_hat, n_ecl)))
    i_calc.append(i_k)

    # Ángulo entre el plano invariable y el plano orbital del planeta k
    dt = np.rad2deg(np.arccos(np.dot(Lt_hat, L_k_hat)))
    delta_theta.append(dt)

print("b) Comparación de planos\n")
print(f"{'Plano':<35} {'Inclinación respecto a la eclíptica':>38} {'Δθ respecto al plano invariable':>32}")
print("-" * 108)
print(f"{'Plano invariable (Sol–Jup–Sat)':<35} {i_inv_deg:>35.4f}°  {'—':>30}")
print(f"{'Órbita de Júpiter (dato dado)':<35} {i_Jup_dado_deg:>35.4f}°  {delta_theta[0]:>29.4f}°")
print(f"{'Órbita de Júpiter (calculada)':<35} {i_calc[0]:>35.4f}°  {'':>30}")
print(f"{'Órbita de Saturno (dato dado)':<35} {i_Sat_dado_deg:>35.4f}°  {delta_theta[1]:>29.4f}°")
print(f"{'Órbita de Saturno (calculada)':<35} {i_calc[1]:>35.4f}°  {'':>30}")
print()

# Contribuciones al momentum angular total
L_norms = np.linalg.norm(L, axis=1)
L_total_mag = np.sum(L_norms)
print("Contribución relativa de cada cuerpo al |L| total:")
for i, id in enumerate(bodies):
    print(f"  {id:<25}: {100*L_norms[i]/L_total_mag:.3f}%")

## Interpretación de resultados

### Parte a)
El plano invariable de Laplace del sistema Sol–Júpiter–Saturno presenta una inclinación $i_{\text{inv}}$ respecto al plano de la eclíptica. Este ángulo mide cuánto se inclina el plano perpendicular al momentum angular total del subsistema respecto al plano de referencia eclíptico J2000.

### Parte b)
El plano invariable se ubica **entre** los planos orbitales de Júpiter y Saturno:

$$i_J = 1^\circ 18' < i_{\text{inv}} < i_S = 2^\circ 29'$$

Esto es físicamente esperado: el plano invariable es un promedio ponderado por el momentum angular de cada cuerpo. Dado que Júpiter posee una masa mayor y una órbita más cercana al Sol, su contribución al momentum angular total es dominante, por lo que el plano invariable se encuentra **más cercano al plano orbital de Júpiter** que al de Saturno.

Además, $\Delta\theta_k$ cuantifica el ángulo entre el plano invariable y el plano orbital de cada planeta:
- $\Delta\theta_J < \Delta\theta_S$, confirmando que el plano invariable está más alineado con Júpiter.

Este resultado es análogo al del Sistema Solar completo, donde el plano de Laplace queda muy próximo al plano orbital de Júpiter (el planeta más masivo), con una inclinación de tan solo $\sim 0.32^\circ$ respecto a dicho plano.

---

## Resumen

| Plano | Inclinación respecto a la eclíptica |
|-------|-------------------------------------|
| Plano invariable (Sol–Júpiter–Saturno) | $i_{\text{inv}}$ (calculado arriba) |
| Órbita de Júpiter | $1^\circ 18' = 1.300^\circ$ |
| Órbita de Saturno | $2^\circ 29' \approx 2.483^\circ$ |

El plano invariable del sistema Sol–Júpiter–Saturno se sitúa entre ambos planos orbitales, más cercano al de Júpiter dado que este planeta domina el momentum angular del subsistema.

---